# Critical initialization — reading notes and numerical checks

Working notebook for **Pachitariu, Zhong, Gracias, Minisi, Lopez & Stringer (2026),
"A critical initialization for biological neural networks", *Nature*.**
Plan and open questions are in [critical-init-handoff.md](critical-init-handoff.md) and
[CLAUDE.md](CLAUDE.md); the paper's own code is at https://github.com/mouseland/critical_init.

**The model.** Linear dynamics with independent noise,

$$\tau \,\dot{\mathbf{x}} = -\mathbf{x} + A\mathbf{x} + \boldsymbol{\epsilon}
            = M\mathbf{x} + \boldsymbol{\epsilon}, \qquad M = A - I,$$

with $A$ a random symmetric matrix, mean-subtracted and scaled so its largest eigenvalue
is $c \le 1$ (the paper uses $c = 0.998$ and calls this "critically normalized"). Because $A$ is
symmetric, $A$, $M$ and the stationary covariance $\Sigma = \tfrac12 (I-A)^{-1}$ share
eigenvectors, and

$$\lambda_M = \lambda_A - 1, \qquad \lambda_\Sigma = \frac{1}{2(1-\lambda_A)} = -\frac{1}{2\lambda_M}.$$

**Goal.** Reproduce the paper's core spectral claims — the $n^{-2/3}$ decay of $\lambda_\Sigma$
for symmetric $A$, and the way it degrades when $c < 1$ (Fig. 2g–i) — and stress-test the
points where the argument may be fragile (windowing of the power-law fit; symmetry vs.
non-normality). Each step below is one self-contained piece of that.

**How to use this notebook.** Each step is a markdown cell (what and why) followed by
`# [Cell S.C]` code cells that define functions and then apply them in a demo call at the
bottom. The demo call's inputs are the `UPPER_CASE` constants just above it — edit those and
re-run the cell; figures are drawn into a fixed figure number so the re-run replaces the
previous window rather than opening a new one. Cell 0.1 sets up imports and the plotting style.


In [4]:
# [Cell 0.1] --- setup: imports, autoreload, plotting style
#
# Nothing analysis-specific here. natureStylePlots lives in ./utilities.py
# (copied from pyUberPhys so this repo is self-contained).

%load_ext autoreload
%autoreload 2

from typing import NamedTuple, Sequence

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.figure import Figure

from utilities import natureStylePlots

%matplotlib qt

natureStylePlots(fontSizeAdjust="large")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Step 1 — Toy edge spectrum: what sub-critical scaling does to the eigenvalues

Before touching any actual random matrix, look at an idealized version of its top eigenvalues.
For a symmetric random matrix with spectral radius 1, the eigenvalue density near the top edge
vanishes like $\sqrt{1-\lambda}$ (Wigner semicircle). Integrating that density from $\lambda$
up to 1 gives the *rank* $i$ of the eigenvalue at $\lambda$:

$$ i(\lambda) \approx \frac{4\sqrt2\,N}{3\pi}\,(1-\lambda)^{3/2}
   \quad\Longrightarrow\quad
   \lambda_{A,i} \approx 1 - \lambda_0\, i^{2/3}, \qquad
   \lambda_0 = \Big(\frac{3\pi}{4\sqrt2\,N}\Big)^{2/3} . $$

So the top eigenvalues descend from 1 with gaps of order $N^{-2/3}$, not $N^{-1}$ — the
density is thin at the edge. $\lambda_0$ is fixed by $N$, not chosen (the cell prints it).
This is only an edge approximation; it stops being accurate well before $i \sim N$, so the
high-rank tail of these plots should not be taken literally — a real sampled $A$ (Step 2)
will differ there.

Now scale: $A \to cA$. The eigenvalues of $cA$ are just $c\,\lambda_{A,i}$, but the leak
$-I$ in $M = cA - I$ is **not** scaled, so

$$ -\lambda_{M,i} = 1 - c\,\lambda_{A,i} = (1-c) + c\,\lambda_0\, i^{2/3}. $$

The $(1-c)$ term is a floor. For ranks where $c\lambda_0 i^{2/3} \ll 1-c$ all modes decay at
the same rate $\approx 1-c$ (and so have the same variance $\approx 1/(2(1-c))$) — the spectrum is
flat. Above the knee $i^* \sim ((1-c)/\lambda_0)^{3/2}$ the $i^{2/3}$ growth takes over and
the $-2/3$ power law in $\lambda_\Sigma$ appears. Scaling $\tau$ instead would scale both
terms together and leave the shape untouched; that asymmetry is the whole point.

**Cell 1.1** builds this toy spectrum and plots, for each $c$, the top-ranked eigenvalues of
$cA$, of $M = cA - I$ (as $-\lambda_M$, since $\lambda_M<0$), and of
$\Sigma = \tfrac12 (I - cA)^{-1}$, side by side on log–log axes with one point per eigenvalue.
Knobs: `TOY_N` (matrix size, which sets $\lambda_0$), `TOY_C_VALUES` (the spectral radii to
compare), `TOY_MAX_RANK` (how far down the spectrum to show). What to look for: on log–log
axes the critical $c = 1$ curves are straight lines of slope $+2/3$ ($-\lambda_M$) and $-2/3$
($\lambda_\Sigma$); each $c < 1$ curve is flat at its floor and bends onto the critical line
above its knee. A knee inside the paper's rank 10–500 fit window is what lowers the fitted
exponent in Fig. 2i. The three panels are the same ranks seen through the three matrices: a
small shift in $\lambda_A$ becomes a large change in the top $\lambda_M$ and $\lambda_\Sigma$
because those depend on the *gap* $1 - c\lambda_A$.


In [23]:
# [Cell 1.1] --- Step 1: toy edge spectrum, and the effect of scaling A by c < 1
#
# Builds the idealized top-edge eigenvalues lambda_i = 1 - lambda0 * i^(2/3) of a
# critically normalized symmetric A, then plots the top-ranked eigenvalues of cA,
# of M = cA - I, and of Sigma = (I - cA)^-1 / 2 for each c. No random matrices
# yet -- that's Step 2.
#
#   edgeLambda0()          the semicircle edge constant lambda0(N) = (3 pi / (4 sqrt2 N))^(2/3)
#   toyEdgeEigenvalues()   the toy spectrum lambda_i = 1 - lambda0 i^(2/3), i = 1..N, descending
#   plotToySpectra()       three log-log panels (cA, -M, Sigma) of eigenvalue vs rank, points + lines, per c
#   TOY_N                  number of eigenvalues in the toy spectrum
#   TOY_C_VALUES           the spectral radii c swept in the demo call
#   TOY_MAX_RANK           highest rank shown in the demo call


def edgeLambda0(N: int) -> float:
    """
    The constant lambda0 in the edge law 1 - lambda_i = lambda0 * i^(2/3).

    Derived by integrating the semicircle density (2N/pi) sqrt(1 - lambda^2),
    approximated near lambda = 1 as (2N/pi) sqrt(2) sqrt(1 - lambda), from
    lambda up to 1 and inverting the resulting rank-vs-value relation.

    Parameters
    ----------
    N : int
        Number of eigenvalues (matrix size). Must be positive.

    Returns
    -------
    float
        lambda0 = (3 pi / (4 sqrt(2) N))^(2/3); positive, ~3e-3 for N = 1e4.

    Raises
    ------
    ValueError
        If N is not a positive integer.
    """
    if N < 1:
        raise ValueError(f"edgeLambda0: N must be a positive integer, got {N!r}")
    return (3 * np.pi / (4 * np.sqrt(2) * N)) ** (2 / 3)


def toyEdgeEigenvalues(N: int, *, lambda0: float | None = None) -> np.ndarray:
    """
    Idealized top-edge eigenvalues of a critically normalized symmetric random
    matrix: lambda_i = 1 - lambda0 * i^(2/3) for i = 1..N, sorted descending.

    This is the semicircle *edge approximation* only; it is accurate for
    i << N and is not a valid semicircle quantile deep in the bulk (for the
    default lambda0 the last value is about -0.4, not -1).

    Parameters
    ----------
    N : int
        Number of eigenvalues to generate.
    lambda0 : float or None, default None
        The edge constant. When None, uses `edgeLambda0(N)`, which pins the
        spectrum to the semicircle with N eigenvalues; pass a value to
        override (must be positive).

    Returns
    -------
    np.ndarray
        Shape (N,), the eigenvalues lambda_1 > lambda_2 > ... > lambda_N,
        with lambda_1 = 1 - lambda0 just below 1.

    Raises
    ------
    ValueError
        If lambda0 is given and is not positive.
    """
    if lambda0 is None:
        lambda0 = edgeLambda0(N)
    if lambda0 <= 0:
        raise ValueError(f"toyEdgeEigenvalues: lambda0 must be positive, got {lambda0!r}")
    ranks = np.arange(1, N + 1, dtype=float)
    return 1.0 - lambda0 * ranks ** (2 / 3)


def plotToySpectra(lamA: np.ndarray,
                   cValues: Sequence[float],
                   *,
                   maxRank: int = 50,
                   figNum: int | str | None = None,
                   ) -> Figure:
    """
    Three-panel figure of the top `maxRank` eigenvalues of cA, of the dynamics
    matrix M = cA - I, and of the stationary covariance Sigma = (I - cA)^-1 / 2,
    one line-with-markers per c, all against rank on log-log axes. The M
    panel shows -lambda_M (lambda_M is negative), so it is the gap 1 - c lambda_A.

    Uses the symmetric-A relations lambda_M = c lambda_A - 1 and
    lambda_Sigma = 1 / (2 (1 - c lambda_A)), so the three panels are the same
    ranks seen through the three matrices.

    Parameters
    ----------
    lamA : np.ndarray
        Eigenvalues of the critically normalized A, shape (N,), descending.
    cValues : sequence of float
        Spectral radii to sweep; each must satisfy 0 < c <= 1.
    maxRank : int, default 50
        Highest rank shown; must be between 1 and lamA.size.
    figNum : int or str or None, default None
        Figure number to draw into. When given, that figure is cleared and
        reused, so re-running the cell replaces the previous plot rather than
        opening another window; when None, a new figure is created.

    Returns
    -------
    fig : matplotlib.figure.Figure
        1x3 figure: eigenvalues of cA, of -M, and of Sigma vs rank 1..maxRank
        (log-log), each with one marker+line series per c; a grey dotted line
        at 1 in the cA panel; legend on the first panel.
        Returned (not closed) so the caller can annotate or save it.

    Raises
    ------
    ValueError
        If any c is outside (0, 1], or maxRank is outside [1, lamA.size].
    """
    cValues = list(cValues)
    if any(c <= 0 or c > 1 for c in cValues):
        raise ValueError(f"plotToySpectra: every c must be in (0, 1], got {cValues!r}")
    if not (1 <= maxRank <= lamA.size):
        raise ValueError(f"plotToySpectra: maxRank must be in [1, {lamA.size}], got {maxRank!r}")

    ranks = np.arange(1, maxRank + 1)
    lamTop = lamA[:maxRank]

    fig = plt.figure(num=figNum, figsize=(15, 4.5))
    fig.clf()
    fig.set_size_inches(15, 4.5)
    axA, axM, axS = fig.subplots(1, 3)
    fig.subplots_adjust(top=0.85, bottom=0.16, left=0.06, right=0.99, wspace=0.32)

    colors = plt.cm.viridis(np.linspace(0.0, 0.75, len(cValues)))
    for c, col in zip(cValues, colors):
        label  = f"c = {c:g}"
        lamCA  = c * lamTop
        lamM   = lamCA - 1.0
        lamSig = 1.0 / (2.0 * (1.0 - lamCA))
        style  = dict(color=col, lw=1.5, marker="o", ms=4, label=label)
        axA.loglog(ranks, lamCA,  **style)
        axM.loglog(ranks, -lamM,  **style)   # lambda_M < 0: plot its magnitude on the log axis
        axS.loglog(ranks, lamSig, **style)

    axA.axhline(1.0, color="0.6", ls=":", lw=1)

    axA.set_title(r"eigenvalues of $cA$")
    axA.set_ylabel(r"$\lambda_{A,i}$ (scaled by $c$)")
    axM.set_title(r"eigenvalues of $M = cA - I$ (sign flipped)")
    axM.set_ylabel(r"$-\lambda_{M,i} = 1 - c\,\lambda_{A,i}$")
    axS.set_title(r"eigenvalues of $\Sigma = \frac{1}{2}(I - cA)^{-1}$")
    axS.set_ylabel(r"$\lambda_{\Sigma,i} = 1\,/\,2(1 - c\,\lambda_{A,i})$")
    for ax in (axA, axM, axS):
        ax.set_xlabel("rank i")
    axA.legend(loc="lower left")

    N = lamA.size
    fig.suptitle(f"Step 1: toy edge spectrum, N = {N:,},  "
                 rf"$\lambda_0$ = {1.0 - lamA[0]:.2e}", y=0.99, fontsize=14)
    return fig


TOY_N        = 100
TOY_C_VALUES = [1.0, 0.97, 0.95, 0.8, 0.5, 0.25]
TOY_MAX_RANK = 50

lamA_toy = toyEdgeEigenvalues(TOY_N)
print(f"lambda0 = {edgeLambda0(TOY_N):.3e};  top three eigenvalues: {lamA_toy[:3]};  last: {lamA_toy[-1]:.3f}")
fig_1_1 = plotToySpectra(lamA_toy, TOY_C_VALUES, maxRank=TOY_MAX_RANK, figNum=11)


lambda0 = 6.523e-02;  top three eigenvalues: [0.93476749 0.89644984 0.86431091];  last: -0.405
